# Imports

In [10]:
import os
import sys
import time
import dotenv
import numpy as np
import pandas as pd

print("Python version: {}". format(sys.version))
print("NumPy version: {}". format(np.__version__))
print("pandas version: {}". format(pd.__version__))

import warnings
warnings.filterwarnings('ignore')
print('-'*25)

Python version: 3.13.5 (main, Jun 12 2025, 12:22:43) [Clang 20.1.4 ]
NumPy version: 2.4.4
pandas version: 3.0.2
-------------------------


In [11]:
dotenv.load_dotenv("../.env")

True

In [ ]:
from pydantic import SecretStr
from langchain.chat_models import init_chat_model
from langchain_openai import OpenAIEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.document_loaders import CSVLoader, TextLoader
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_qdrant import QdrantVectorStore


# 1.0 Load data

In [13]:
DATA_FILE_PATH = "../data/medquad.csv"

In [14]:
med_df = pd.read_csv(DATA_FILE_PATH)

med_df.head()

,question,answer,source,focus_area
0,What is (are) Glaucoma ?,Glaucoma is a group of diseases that can damag...,NIHSeniorHealth,Glaucoma
1,What causes Glaucoma ?,"Nearly 2.7 million people have glaucoma, a lea...",NIHSeniorHealth,Glaucoma
2,What are the symptoms of Glaucoma ?,Symptoms of Glaucoma Glaucoma can develop in ...,NIHSeniorHealth,Glaucoma
3,What are the treatments for Glaucoma ?,"Although open-angle glaucoma cannot be cured, ...",NIHSeniorHealth,Glaucoma
4,What is (are) Glaucoma ?,Glaucoma is a group of diseases that can damag...,NIHSeniorHealth,Glaucoma


In [15]:
# Checking NULL values
display(med_df.isnull().sum())

question       0
answer         5
source         0
focus_area    14
dtype: int64

In [16]:
print(len(med_df))
med_df.dropna(axis=0, inplace=True)
print(len(med_df))

16412
16393


In [17]:
focus_areas = med_df['focus_area'].unique().tolist()
focus_areas.sort()
focus_areas

['11-beta-hydroxylase deficiency',
 '15q11.2 microdeletion',
 '15q13.3 microdeletion',
 '15q13.3 microdeletion syndrome',
 '15q13.3 microduplication syndrome',
 '15q24 microdeletion',
 '16p11.2 deletion syndrome',
 '16q24.3 microdeletion syndrome',
 '17 alpha-hydroxylase/17,20-lyase deficiency',
 '17-alpha-hydroxylase deficiency',
 '17-beta hydroxysteroid dehydrogenase 3 deficiency',
 '17q23.1q23.2 microdeletion syndrome',
 '18 Hydroxylase deficiency',
 '18q deletion syndrome',
 '19p13.12 microdeletion syndrome',
 '1p36 deletion syndrome',
 '1q21.1 microdeletion',
 '1q21.1 microdeletion syndrome',
 '1q44 microdeletion syndrome',
 '2,4-Dienoyl-CoA reductase deficiency',
 '2-hydroxyglutaric aciduria',
 '2-methyl-3-hydroxybutyric aciduria',
 '2-methylbutyryl-CoA dehydrogenase deficiency',
 '2009 H1N1 Flu',
 '20p12.3 microdeletion syndrome',
 '21-hydroxylase deficiency',
 '22q11.2 deletion syndrome',
 '22q11.2 duplication',
 '22q11.2 duplication syndrome',
 '22q13.3 deletion syndrome',
 '2

# 2.0 Retrieval Augmented Generation

I want to stick to a RAG with a narrow focus on Nutrition.

In [ ]:
llm = init_chat_model("gpt-4o-mini", model_provider="openai")
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# llm = init_chat_model(
#     "google_genai:" + "gemini-2.5-flash-lite",
#     model_provider="google_genai",
#     api_key=os.environ['GOOGLE_API_KEY']
# )
# embeddings = GoogleGenerativeAIEmbeddings(
#     model="gemini-embedding-001",
#     api_key=SecretStr(os.environ['GOOGLE_API_KEY']),
# )

In [ ]:
# with open('./focus_areas.txt', 'w') as f:
#     f.write("\n".join(focus_areas))

# docs = TextLoader('./focus_areas.txt').load()
# print(docs[:2])

df = pd.DataFrame.from_dict({'focus_area': focus_areas})
df.to_csv('../data/focus_areas.csv', index=False)
docs = CSVLoader('../data/focus_areas.csv').load()
print(docs[:5])

[Document(metadata={'source': './focus_areas.csv', 'row': 0}, page_content='focus_area: 11-beta-hydroxylase deficiency'), Document(metadata={'source': './focus_areas.csv', 'row': 1}, page_content='focus_area: 15q11.2 microdeletion'), Document(metadata={'source': './focus_areas.csv', 'row': 2}, page_content='focus_area: 15q13.3 microdeletion'), Document(metadata={'source': './focus_areas.csv', 'row': 3}, page_content='focus_area: 15q13.3 microdeletion syndrome'), Document(metadata={'source': './focus_areas.csv', 'row': 4}, page_content='focus_area: 15q13.3 microduplication syndrome')]


In [ ]:
vector_store = InMemoryVectorStore.from_documents(docs, embeddings)

Let's use Vector Embeddings to collect the focus areas we want i.e. those topics most semantically similar to Nutrition.

In [18]:
docs = vector_store.similarity_search("Nutrition", k=150)
for doc in docs:
    print(doc)

page_content='focus_area: Nutrition' metadata={'source': './focus_areas.csv', 'row': 2978}
page_content='focus_area: Nutrition for Seniors' metadata={'source': './focus_areas.csv', 'row': 2981}
page_content='focus_area: Nutritional Support' metadata={'source': './focus_areas.csv', 'row': 2982}
page_content='focus_area: Child Nutrition' metadata={'source': './focus_areas.csv', 'row': 793}
page_content='focus_area: Toddler Nutrition' metadata={'source': './focus_areas.csv', 'row': 4093}
page_content='focus_area: Malnutrition' metadata={'source': './focus_areas.csv', 'row': 2540}
page_content='focus_area: Nutrition for Advanced Chronic Kidney Disease in Adults' metadata={'source': './focus_areas.csv', 'row': 2979}
page_content='focus_area: Nutrition for Early Chronic Kidney Disease in Adults' metadata={'source': './focus_areas.csv', 'row': 2980}
page_content='focus_area: Pregnancy and Nutrition' metadata={'source': './focus_areas.csv', 'row': 3387}
page_content='focus_area: Infant and New

In [19]:
topics_chosen = [doc.page_content.split("focus_area: ")[1] for doc in docs]
topics_chosen

['Nutrition',
 'Nutrition for Seniors',
 'Nutritional Support',
 'Child Nutrition',
 'Toddler Nutrition',
 'Malnutrition',
 'Nutrition for Advanced Chronic Kidney Disease in Adults',
 'Nutrition for Early Chronic Kidney Disease in Adults',
 'Pregnancy and Nutrition',
 'Infant and Newborn Nutrition',
 'Diabetic Diet',
 'Dietary Proteins',
 'Dietary Fiber',
 'Vitamins',
 'Vegetarian Diet',
 'DASH Diet',
 'Diets',
 'Eating Disorders',
 'Dietary Fats',
 'Obesity',
 'Dietary Supplements',
 'Weight Control',
 'Vitamin A',
 'Healthy Living',
 'Healthy Aging',
 'Prevent diabetes problems: Keep your nervous system healthy',
 'Vitamin C',
 'Metabolic Disorders',
 'B Vitamins',
 'Folic Acid',
 'Obesity in Children',
 'Food Labeling',
 'Vitamin D',
 'Food Allergy',
 'Antioxidants',
 'Overweight and Obesity',
 'Breastfeeding',
 'Kidney Failure: Eat Right to Feel Right on Hemodialysis',
 'Iron-Deficiency Anemia',
 'Digestive Diseases',
 'Vitamin E',
 "Seniors' Health",
 'Fluid and Electrolyte Balanc

In [20]:
with open('../data/topics_chosen.txt', 'w') as f:
    f.write("\n".join(topics_chosen))

Narrow down the original Q & A dataset to only include these 150 topics similar/related to Nutrition.

In [21]:
small_med_df = med_df[med_df['focus_area'].isin(topics_chosen)]
small_med_df

,question,answer,source,focus_area
103,What is (are) Diabetes ?,Too Much Glucose in the Blood Diabetes means y...,NIHSeniorHealth,Diabetes
104,Who is at risk for Diabetes? ?,"Diabetes is a serious, life-long disease. It c...",NIHSeniorHealth,Diabetes
105,How to prevent Diabetes ?,The two most common forms of diabetes are type...,NIHSeniorHealth,Diabetes
106,What are the symptoms of Diabetes ?,"Diabetes is often called a ""silent"" disease be...",NIHSeniorHealth,Diabetes
107,What are the treatments for Diabetes ?,"Diabetes cannot be cured, but it can be manage...",NIHSeniorHealth,Diabetes
...,...,...,...,...
16320,How to diagnose What I need to know about Diar...,"To find the cause of diarrhea, the health care...",NIDDK,What I need to know about Diarrhea
16321,What are the treatments for What I need to kno...,"Diarrhea is treated by replacing lost fluids, ...",NIDDK,What I need to know about Diarrhea
16322,What to do for What I need to know about Diarr...,"To prevent dehydration when you have diarrhea,...",NIDDK,What I need to know about Diarrhea
16323,How to prevent What I need to know about Diarr...,Two types of diarrhea can be preventedrotaviru...,NIDDK,What I need to know about Diarrhea


Now let's delete the loaded topic documents in the Vector Store.

In [22]:
del vector_store

In [23]:
small_med_df.to_csv('./small_med_df.csv', index=False)
docs = CSVLoader('./small_med_df.csv').load()
print(docs[:5])

[Document(metadata={'source': './small_med_df.csv', 'row': 0}, page_content="question: What is (are) Diabetes ?\nanswer: Too Much Glucose in the Blood Diabetes means your blood glucose (often called blood sugar) is too high. Your blood always has some glucose in it because your body needs glucose for energy to keep you going. But too much glucose in the blood isn't good for your health. Glucose comes from the food you eat and is also made in your liver and muscles. Your blood carries the glucose to all of the cells in your body. Insulin is a chemical (a hormone) made by the pancreas. The pancreas releases insulin into the blood. Insulin helps the glucose from food get into your cells. If your body does not make enough insulin or if the insulin doesn't work the way it should, glucose can't get into your cells. It stays in your blood instead. Your blood glucose level then gets too high, causing pre-diabetes or diabetes. Types of Diabetes There are three main kinds of diabetes: type 1, ty

## 2.0.0 Load the Question and Answer values as documents into the Vector Database

In [ ]:
df = pd.DataFrame.from_dict({'focus_area': focus_areas})
df.to_csv('../data//focus_areas.csv', index=False)
docs = CSVLoader('../data/focus_areas.csv').load()
print(docs[:5])

[Document(metadata={'source': '../data/focus_areas.csv', 'row': 0}, page_content='focus_area: 11-beta-hydroxylase deficiency'), Document(metadata={'source': '../data/focus_areas.csv', 'row': 1}, page_content='focus_area: 15q11.2 microdeletion'), Document(metadata={'source': '../data/focus_areas.csv', 'row': 2}, page_content='focus_area: 15q13.3 microdeletion'), Document(metadata={'source': '../data/focus_areas.csv', 'row': 3}, page_content='focus_area: 15q13.3 microdeletion syndrome'), Document(metadata={'source': '../data/focus_areas.csv', 'row': 4}, page_content='focus_area: 15q13.3 microduplication syndrome')]


In [25]:
vector_store = QdrantVectorStore.from_documents(
    docs,
    embeddings,
    url=os.environ['QDRANT_URL'],
    prefer_grpc=True,
    api_key=os.environ['QDRANT_API_KEY'],
    collection_name="test",
)

In [32]:
docs = vector_store.similarity_search("What is type 1 diabetes?", k=2)
for doc in docs:
    print(doc)

page_content='focus_area: type 1 diabetes' metadata={'source': '../data/focus_areas.csv', 'row': 5108, '_id': 'e2751373-f497-4efa-9829-69d61d065e67', '_collection_name': 'test'}
page_content='focus_area: Diabetes Type 1' metadata={'source': '../data/focus_areas.csv', 'row': 1186, '_id': '16897d25-874d-4023-a4fb-d2e9f51a3f9d', '_collection_name': 'test'}


In [33]:
del vector_store

In [34]:
docs = CSVLoader('../data/medquad.csv').load()
print(docs[:5])

[Document(metadata={'source': '../data/medquad.csv', 'row': 0}, page_content="question: What is (are) Glaucoma ?\nanswer: Glaucoma is a group of diseases that can damage the eye's optic nerve and result in vision loss and blindness. While glaucoma can strike anyone, the risk is much greater for people over 60. How Glaucoma Develops  There are several different types of glaucoma. Most of these involve the drainage system within the eye. At the front of the eye there is a small space called the anterior chamber. A clear fluid flows through this chamber and bathes and nourishes the nearby tissues. (Watch the video to learn more about glaucoma. To enlarge the video, click the brackets in the lower right-hand corner. To reduce the video, press the Escape (Esc) button on your keyboard.) In glaucoma, for still unknown reasons, the fluid drains too slowly out of the eye. As the fluid builds up, the pressure inside the eye rises. Unless this pressure is controlled, it may cause damage to the op

In [ ]:
# vector_store = QdrantVectorStore.from_documents(
#     docs,
#     embeddings,
#     url=os.environ['QDRANT_URL'],
#     prefer_grpc=True,
#     api_key=os.environ['QDRANT_API_KEY'],
#     collection_name="medquad",
# )

In [ ]:
# del vector_store

In [40]:
docs = vector_store.similarity_search("What is type 1 diabetes?", k=2)
for doc in docs:
    print(doc)

page_content='question: What is (are) Diabetes mellitus type 1 ?
answer: Diabetes mellitus type 1 (DM1) is a condition in which cells in the pancreas (beta cells) stop producing insulin, causing abnormally high blood sugar levels. Lack of insulin results in the inability of the body to use glucose for energy and control the amount of sugar in the blood. DM1 can occur at any age, but usually develops by early adulthood, most often in adolescence. Symptoms of high blood sugar may include frequent urination, excessive thirst, fatigue, blurred vision, tingling or loss of feeling in the hands and feet, and weight loss. The exact cause of DM1 is unknown, but having certain "variants" of specific genes may increase a person's risk to develop the condition. A predisposition to develop DM1 runs in families, but no known inheritance pattern exists. Treatment includes blood sugar control and insulin replacement therapy. Improper control can cause recurrence of high blood sugar, or abnormally low 

In [9]:
vector_store = QdrantVectorStore.from_existing_collection(
    embedding=embeddings,
    collection_name="medquad",
    url=os.environ['QDRANT_URL'],
    api_key=os.environ['QDRANT_API_KEY'],
)

# 2.1 Generate RAG prompt with context

In [ ]:
from pydantic import BaseModel
from typing import List, Dict, Optional

# from langchain import hub
from langchain_core.documents import Document
from langchain_core.vectorstores import VectorStore
from langchain_core.prompts import PromptTemplate

# prompt = hub.pull("rlm/rag-prompt")
# print("System Prompt we are using: ")
# print(prompt.messages[0].prompt.template)

SYSTEM_PROMPT = """
You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. 
If you don't know the answer, just say that you don't know. Keep the answer short and concise. Try to use less than 5 sentences.
"""
PROMPT_TEMPLATE = SYSTEM_PROMPT + """
Question: {question} 
Context: {context} 
Answer: 
"""
prompt = PromptTemplate.from_template(PROMPT_TEMPLATE)

class State(BaseModel):
    question: str
    context: Optional[List[Document]]
    answer: str
    tenant: Optional[str]


def retrieve(db: VectorStore, state: State):
    # defaults to k = 4, so max 4 documents are returned.
    retrieved_docs = db.similarity_search(state.question)
    return {"context": retrieved_docs}


def generate(state: State):
    if state.context is None:
        state.context = []
    docs_content = "\n\n".join(doc.page_content for doc in state.context)
    messages = prompt.invoke(
        {"question": state.question, "context": docs_content})
    response = llm.invoke(messages)
    return {"answer": response.content}


def retrieve_and_generate(user_prompt, tenant=None):
    state = State(question=user_prompt, context=None, answer='', tenant=None)
    if tenant:
        state.tenant = tenant

    context = retrieve(vector_store, state)
    state.context = context['context']

    ans = generate(state)['answer']
    return ans


def print_rag(user_prompt, rag_answer):
    print("User: \n{}".format(user_prompt))
    print()
    print("Answer: \n{}".format(rag_answer))


def print_rag2(user_prompt, rag_answer):
    print("Question: \t{}".format(user_prompt))
    print("Answer: \t{}".format(rag_answer))


def print_rag3(user_prompt, rag_answer):
    tmp_df = pd.DataFrame.from_dict(
        {'User': [user_prompt], 'RAG AI Response': [rag_answer]})
    with pd.option_context('display.max_colwidth', None, "colheader_justify", "left"):
        display(tmp_df)
    return tmp_df

System Prompt we are using: 
You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.
Question: {question} 
Context: {context} 
Answer:


# 2.2 Ask Questions

In [12]:
ques = "Tell me about the types of diabetes..."
ans = retrieve_and_generate(ques)
print_rag(ques, ans)

User: 
Tell me about the types of diabetes...

Answer: 
The main types of diabetes include type 1, type 2, and gestational diabetes. Type 1 diabetes is an autoimmune condition where the body does not produce insulin, while type 2 is characterized by insulin resistance and is more common, often linked to lifestyle factors. Gestational diabetes occurs during pregnancy and typically resolves after childbirth, though it can increase the risk of developing type 2 diabetes later on.


## Pain point 1: Retrieval Configuration
The Policy/Configuration for retrieval, defining things such as how many document are returned, can greatly affect the output.

Imagine there were 30 types of diabetes, and our Q&A dataset did not contain any documents which mentioned them. Only individual documents which talk about individual types of diabetes.

Our Vector DB Search yields 4 documents as a good middle ground between returning too few documents and too many, making the prompt very long, possibly long enough that the question is "forgotten".

Data that's provided to the RAG system matters a lot!

Configuration that's used matters a lot!

## Pain point 2: Evaluation and Improvement 

Is this RAG good enough?

How do you determine that?

I'm learning about this topic, and I'll keep sharing any knowledge or resources I come across in my notebooks.

## Pain point 3: Processing Unstructured Data

In this case, the dataset is highly structured, and we have high quality data -- question and answer pairs.

This makes our life easy, but if we had an unstructured dataset, we would have to do a lot of clean-up, NER, summarization, classification, etc.

If we wanted to create Question and Answer pairs from Doctor's notes where patients asked questions and an AI transcribed the visit live, we would need to do many of the steps mentioned above.

In [23]:
ques = "I eat too much fried food, is that bad?"
ans = retrieve_and_generate(ques)
print_rag(ques, ans)

User: 
I eat too much fried food, is that bad?

Answer: 
Yes, eating too much fried food can be bad for your health, as it often contains unhealthy fats, particularly trans fats, which can raise cholesterol levels and increase the risk of heart disease. It is important to consider healthier fat options and moderation in your diet. Reducing fried foods can help improve your overall health and decrease the risk of related conditions.


In [24]:
ques = "How much water should a person drink in a day?"
ans = retrieve_and_generate(ques)
print_rag(ques, ans)

User: 
How much water should a person drink in a day?

Answer: 
An average person needs about 3 quarts of water daily, but this can vary based on activity level and environmental conditions. It's important for individuals, especially elderly and young children, to monitor their fluid intake more closely. Those on hemodialysis should consult with their dietitian for personalized fluid allowances.


## RAG is too easy?

The easy part is what's done in this notebook.

The hard part is Retrieval configuration, Processing Data, and setting up evaluation pipelines, metrics logging, etc.

Making decisions when taking your product to market and reacting to how your clients use the product.

Being Agile is important.

In [25]:
ques = "Can you prescribe me some medicine for my stomach ulcer?"
ans = retrieve_and_generate(ques)
print_rag(ques, ans)

User: 
Can you prescribe me some medicine for my stomach ulcer?

Answer: 
I can't prescribe medicine, but it's important to see a doctor for proper evaluation and treatment for your stomach ulcer. Common treatments for ulcers typically include medications that reduce stomach acid and promote healing. Over-the-counter options may not be sufficient for more serious conditions like peptic ulcers.


## Uh-oh
Luckily OpenAI's models have been taught to not prescribe medicines, lest someone sue them over it.

But, if you used an open source LLM, you might need to engineer your prompts and use other techniques to prevent Level 1 easy lawsuits, especially before precedents have been set.

Even a simple "Don't prescribe any medicines under any circumstances" added to the system prompt might work.

Cestonaro C, Delicati A, Marcante B, Caenazzo L, Tozzo P. Defining medical liability when artificial intelligence is applied on diagnostic algorithms: a systematic review. Front Med (Lausanne). 2023 Nov 27;10:1305756. doi: 10.3389/fmed.2023.1305756. PMID: 38089864; PMCID: PMC10711067.
https://pmc.ncbi.nlm.nih.gov/articles/PMC10711067/#S5

## Domain Knowledge

Domain knowledge will be important to figure out what's acceptable for the AI to output, and what's not.

Understanding industry specific jargon will also allow data engineers to better pre-process the input data.

In [26]:
ques = "What is wasting?"
ans = retrieve_and_generate(ques)
print_rag(ques, ans)

User: 
What is wasting?

Answer: 
Wasting is a form of malnutrition characterized by significant weight loss and muscle wasting, often due to inadequate nutrition or illness. It typically suggests a severe lack of essential nutrients and energy, and can lead to serious health complications. Addressing the underlying causes and restoring proper nutrition are crucial for recovery.


## Retrieval Augmented Generation Challenges
In general for domain specific RAG (some of these may apply to this dataset).

### Retrieval Component
- Relevance of retrieved documents
    - Issue: Irrelevant or outdated medical literature or answers.
    - Causes: Weak embedding models, Synonym mismatches like "heart attack" vs. "myocardial infarction", outdated knowledge like "saturated fats are bad for you" without any nuance.
    - Mitigation: Domain specific embeddings (like those used in BioBERT or ClinicalBERT), expanding queries with synonyms, regularly updating knowledge base.
- Coverage
    - Issue: Gaps in the database, or biases in source data.
    - Mitigation: Audit the dataset for diversity in information and inclusion of rare diseases etc, Augment with specialized datasets (rare diseases, misconceptions and corrections etc).
### Generation Component
- Hallucinations
    - Issue: Plausible sounding, but incorrect and un-supported claims (inventing drug dosages, inventing drug names etc).
    - Mitigation: Constrain outputs to retrieved sources, Confidence scores to flag un-certain answers, Rule-based checks (dosage ranges).
- Ambiguity
    - Issue: Queries like "What causes fatigue?" could relate to several factors.
    - Mitigation: Implement Follow-up questions, Return ranked diagnoses with probabilities.
### Data Quality & Ethics
- Outdated or conflicting information
    - Issue: Rapidly evolving medical knowledge, Conflicting studies in the database.
    - Mitigation: Flag conflicting evidence and highlight consensus guildelines, Prioritize peer-reviewed sources, and meta-analyses.
- Privacy and Bias
    - Issue: Sensitive patient data in corpus or biased recommendations.
    - Mitigation: De-identify training data, Test for bias (remember Facebook's face recognition AIs which only detected white faces? That was due to lack of training data for POC, given the sources of their data was mostly North America).
### Evaluation Challenges
- Lack of ground truth
- Clinical safety
    - Mitigation: Human-in-the-loop validation for high risk queries, Guardrails to block unsafe outputs (drug interaction checking, ask follow-ups before finalizing prescriptions).

There are more challenges such as user interaction challenges, challenges in how you present the data, scalability etc.

In [27]:
small_med_df.sample(10)

,question,answer,source,focus_area
2276,What is (are) Malabsorption Syndromes ?,Your small intestine does most of the digestin...,MPlusHealthTopics,Malabsorption Syndromes
551,What is (are) High Blood Cholesterol ?,Too much cholesterol in your blood is called h...,NIHSeniorHealth,High Blood Cholesterol
16064,What is (are) Nutrition for Early Chronic Kidn...,"As blood pressure rises, the risk of damage to...",NIDDK,Nutrition for Early Chronic Kidney Disease in ...
16268,What is (are) Diabetic Kidney Disease ?,Diabetes is a complex group of diseases with a...,NIDDK,Diabetic Kidney Disease
2166,What is (are) Health Checkup ?,Regular health exams and tests can help find p...,MPlusHealthTopics,Health Checkup
15769,What to do for Anemia of Inflammation and Chro...,"People with anemia caused by iron, vitamin B12...",NIDDK,Anemia of Inflammation and Chronic Disease
8178,What is (are) Anemia ?,Espaol\n \nAnemia (uh-NEE-me-uh...,NHLBI,Anemia
16322,What to do for What I need to know about Diarr...,"To prevent dehydration when you have diarrhea,...",NIDDK,What I need to know about Diarrhea
15766,What are the symptoms of Anemia of Inflammatio...,Anemia of inflammation and chronic disease typ...,NIDDK,Anemia of Inflammation and Chronic Disease
15765,What causes Anemia of Inflammation and Chronic...,Anemia of inflammation and chronic disease is ...,NIDDK,Anemia of Inflammation and Chronic Disease


## More questions & answers

In [30]:
ques = "How often should I get a health check-up?"
ans = retrieve_and_generate(ques)
print_rag(ques, ans)

User: 
How often should I get a health check-up?

Answer: 
You should get regular health check-ups, but the frequency depends on your age, health, family history, and lifestyle. For many adults, it's recommended to have a check-up at least once a year. Children should also receive regular check-ups to monitor their development and catch potential issues early.


In [31]:
ques = "I have been feeling weak and dizzy ever since a mosquito bit me a month ago..."
ans = retrieve_and_generate(ques)
print_rag(ques, ans)

User: 
I have been feeling weak and dizzy ever since a mosquito bit me a month ago...

Answer: 
Feeling weak and dizzy could be indicative of anemia, which commonly presents symptoms such as fatigue, dizziness, and shortness of breath. It's important to consult a healthcare provider for an accurate diagnosis, especially since these symptoms have persisted since the mosquito bite. They can evaluate your condition and determine if there is an underlying issue that needs to be addressed.


In [32]:
ques = "Do you think I could be anemic?"
ans = retrieve_and_generate(ques)
print_rag(ques, ans)

User: 
Do you think I could be anemic?

Answer: 
If you are experiencing symptoms like fatigue, weakness, or dizziness, it could be a sign of anemia, but only a doctor can diagnose it. They will evaluate your medical history, symptoms, and may conduct tests, such as a complete blood count (CBC), to determine if you are anemic. It's best to consult your healthcare provider for an accurate diagnosis.


In [33]:
ques = "What's a blood count?"
ans = retrieve_and_generate(ques)
print_rag(ques, ans)

User: 
What's a blood count?

Answer: 
A blood count, specifically a complete blood count (CBC), is a common test used to measure various components of blood, including red blood cells, white blood cells, and platelets. It provides important measurements like hemoglobin and hematocrit levels, which help diagnose conditions like anemia. The CBC can indicate overall health and detect a wide range of disorders, including infections and blood cancers.


In [34]:
ques = "Thanks"
ans = retrieve_and_generate(ques)
print_rag(ques, ans)

User: 
Thanks

Answer: 
You're welcome! If you have any more questions or need further information, feel free to ask.
